In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.data_processing import format_context_with_boundaries
from utils.ipa_processing import syllabify_ipa
SEP = "_"

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri75.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
df = format_context_with_boundaries(df, lang="ae")

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.85,1.87,1.87,2.04,ð,ɛ,ð,then,sub043_2drt_10_northwind2_r1_video.mp4_0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,2.04,2.07,ɛ,n,ð_ɛ,then,sub043_2drt_10_northwind2_r1_video.mp4_0,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,2.09,2.14,ð,ʌ,ð,the,sub043_2drt_10_northwind2_r1_video.mp4_1,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,2.18,2.24,n,ɔ,n,north,sub043_2drt_10_northwind2_r1_video.mp4_2,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,2.24,2.39,ɔ,ɹ,n_ɔ,north,sub043_2drt_10_northwind2_r1_video.mp4_2,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
26851,52.92,52.97,52.97,53.01,æ,ŋ,l_æ,language,sub071_2drt_07_grandfather1_r1_video.mp4_134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26852,52.97,53.01,53.01,53.04,ŋ,g,l_æ_ŋ,language,sub071_2drt_07_grandfather1_r1_video.mp4_134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26853,53.01,53.04,53.04,53.10,g,w,l_æ_ŋ_g,language,sub071_2drt_07_grandfather1_r1_video.mp4_134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26854,53.04,53.10,53.10,53.28,w,ɨ,l_æ_ŋ_g_w,language,sub071_2drt_07_grandfather1_r1_video.mp4_134,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [4]:
with open("../../pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae").replace(".", "_._"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for phone1, phone2 in tqdm.tqdm(zip(df["phone1"], df["phone2"]), total=len(df)):
    bigrams.add(phone1+SEP+phone2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    phones = pronunciation.split(SEP)
    for phone in phones:
        if phone not in monogram_counts:
            monogram_counts[phone] = 0
        monogram_counts[phone] += count

100%|██████████| 48576/48576 [00:00<00:00, 1015407.40it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

100%|██████████| 48576/48576 [00:00<00:00, 176688.80it/s]


In [7]:
data = {
    "bigram_surprisal": [],
    "phone2_pos_entropy": [],
}

In [8]:
data["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["phone1"] + SEP + x["phone2"]), axis=1).tolist()
data["phone2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1).tolist()
data["phone1_freq"] = df.apply(lambda x: monogram_counts.get(x["phone1"], None), axis=1).tolist()
data["phone2_freq"] = df.apply(lambda x: monogram_counts.get(x["phone2"], None), axis=1).tolist()

In [9]:
pd.DataFrame(data).to_csv("surprisal_aemri75.csv", index=False)